In [ ]:
import numpy as np
import tensorflow as tf
from keras.models import Sequential
from keras.layers import Dense, Activation, GRU
from keras.optimizers import RMSprop
from keras.callbacks import LambdaCallback, ModelCheckpoint, ReduceLROnPlateau
import random
import sys

In [ ]:
import gdown
!pip install --upgrade --no-cache-dir gdown

In [ ]:
!gdown --id 1soknKGxxEMcpopGvwNwZoOo7akxQDo54

with open('poems.txt', 'r') as file:
    text = file.read()  # Read the entire file content within the 'with' block
print(text[:500])


/usr/local/lib/python3.11/dist-packages/gdown/__main__.py:140: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From: https://drive.google.com/uc?id=1soknKGxxEMcpopGvwNwZoOo7akxQDo54
To: /content/poems.txt
100% 888k/888k [00:00<00:00, 41.6MB/s]
Through the forest deep, where shadows linger long, the night sings its song.
In the stillness of night, the moon whispers its light.
Under the velvet sky, dreams take flight on wings of stars.
A single leaf falls, carried away by the gentle breeze.
In the stillness of night, the moon whispers its light.
Love is the light that guides us through the darkest times.
A single leaf falls, carried away by the gentle breeze.
A whisper of hope in the silence of the dawn.
A whisper of hope in the silence


In [ ]:
vocabulary = sorted(list(set(text)))
char_to_indices = dict((c, i) for i, c in enumerate(vocabulary))
indices_to_char = dict((i, c) for i, c in enumerate(vocabulary))

In [ ]:
max_length = 100
steps = 5
sentences = []
next_chars = []

for i in range(0, len(text) - max_length, steps):
    sentences.append(text[i: i + max_length])
    next_chars.append(text[i + max_length])

X = np.zeros((len(sentences), max_length, len(vocabulary)), dtype=np.bool_)
y = np.zeros((len(sentences), len(vocabulary)), dtype=np.bool_)

for i, sentence in enumerate(sentences):
    for t, char in enumerate(sentence):
        X[i, t, char_to_indices[char]] = 1
    y[i, char_to_indices[next_chars[i]]] = 1

In [ ]:
model = Sequential()
model.add(GRU(128, input_shape=(max_length, len(vocabulary))))
model.add(Dense(len(vocabulary)))
model.add(Activation('softmax'))

optimizer = RMSprop(learning_rate=0.01)
model.compile(loss='categorical_crossentropy', optimizer=optimizer)

/usr/local/lib/python3.11/dist-packages/keras/src/layers/rnn/rnn.py:200: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


##Helper functions to train the network

##a) Helper function to sample the next character:

In [ ]:
def sample_index(preds, temperature=1.0):
    preds = np.asarray(preds).astype('float64')
    preds = np.log(preds) / temperature
    exp_preds = np.exp(preds)
    preds = exp_preds / np.sum(exp_preds)
    probas = np.random.multinomial(1, preds, 1)
    return np.argmax(probas)


Let me quickly explain what it's doing:

preds: This is an array of probabilities (typically a model's output).

temperature: This controls the "randomness" of the sampling:

Low temperature (< 1.0): Makes the model more confident (sharper distribution).

High temperature (> 1.0): Makes the model more random (flatter distribution).

Steps:

Convert to float64 to ensure precision.

Apply log to the probabilities and divide by the temperature — this adjusts the distribution.

Exponentiate to reverse the log but now with adjusted sharpness.

Normalize the probabilities to sum to 1.

Sample from this distribution using a multinomial draw.

Return the index of the chosen sample.

Basically, this function picks an index according to a probability distribution, but temperature lets you control how "bold" or "exploratory" the sampling is.

##b) Helper function for epoch

In [ ]:
def on_epoch_end(epoch, logs):
    print("\n----- Epoch", epoch)

##c) Helper function defining Callbacks

In [ ]:
print_callback = LambdaCallback(on_epoch_end=on_epoch_end)
filepath = "weights.keras"
checkpoint = ModelCheckpoint(filepath, monitor='loss', verbose=1, save_best_only=True, mode='min')
reduce_alpha = ReduceLROnPlateau(monitor='loss', factor=0.2, patience=1, min_lr=0.001)
callbacks = [print_callback, checkpoint, reduce_alpha]

LambdaCallback(on_epoch_end=on_epoch_end): Calls the on_epoch_end function after each training epoch, which generates text.

ModelCheckpoint(filepath, monitor='loss'): Saves the model weights to weights.keras after every epoch if the model’s loss improves.

ReduceLROnPlateau(): Reduces the learning rate by a factor of 0.2 if the loss stops improving for 1 epoch.

In [ ]:
model.fit(X, y, batch_size=128, epochs=30, callbacks=callbacks)

Epoch 1/30
1389/1389 ━━━━━━━━━━━━━━━━━━━━ 0s 275ms/step - loss: 0.6737
----- Epoch 0

Epoch 1: loss improved from inf to 0.23067, saving model to weights.keras
1389/1389 ━━━━━━━━━━━━━━━━━━━━ 384s 275ms/step - loss: 0.6734 - learning_rate: 0.0100
Epoch 2/30
1389/1389 ━━━━━━━━━━━━━━━━━━━━ 0s 272ms/step - loss: 0.0522
----- Epoch 1

Epoch 2: loss improved from 0.23067 to 0.05141, saving model to weights.keras
1389/1389 ━━━━━━━━━━━━━━━━━━━━ 378s 272ms/step - loss: 0.0522 - learning_rate: 0.0100
Epoch 3/30
1389/1389 ━━━━━━━━━━━━━━━━━━━━ 0s 271ms/step - loss: 0.0509
----- Epoch 2

Epoch 3: loss improved from 0.05141 to 0.05051, saving model to weights.keras
1389/1389 ━━━━━━━━━━━━━━━━━━━━ 381s 271ms/step - loss: 0.0509 - learning_rate: 0.0100
Epoch 4/30
1389/1389 ━━━━━━━━━━━━━━━━━━━━ 0s 271ms/step - loss: 0.0713
----- Epoch 3

Epoch 4: loss did not improve from 0.05051
1389/1389 ━━━━━━━━━━━━━━━━━━━━ 376s 271ms/step - loss: 0.0713 - learning_rate: 0.0100
Epoch 5/30
1389/1389 ━━━━━━━━━━━━━━━━━━

model.fit(): Trains the model on the input data (X) and target labels (y) for 30 epochs with a batch size of 128.
callbacks=callbacks: Passes the defined callbacks, which manage actions during training

##Generating new random text

In [ ]:
def generate_text(length, diversity):
    start_index = random.randint(0, len(text) - max_length - 1)
    generated = ''
    sentence = text[start_index: start_index + max_length]
    generated += sentence
    for i in range(length):
        x_pred = np.zeros((1, max_length, len(vocabulary)))
        for t, char in enumerate(sentence):
            x_pred[0, t, char_to_indices[char]] = 1.
        preds = model.predict(x_pred, verbose=0)[0]
        next_index = sample_index(preds, diversity)
        next_char = indices_to_char[next_index]
        generated += next_char
        sentence = sentence[1:] + next_char
    return generated

print(generate_text(500, 0.2))

generate_text(): Generates text of a given length (length) based on the trained model and diversity (temperature).

sample_index(preds, diversity): Samples the next character index based on the predicted probability distribution (preds), controlled by the diversity value.
model.predict(): Predicts the next character by feeding the current sequence into the model.

next_char = indices_to_char[next_index]: Converts the predicted index back to the corresponding character.